<a href="https://colab.research.google.com/github/RagaSandhiya05/LLM-From-Scratch/blob/main/07_Self_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SELF-ATTENTION & CASUAL MASKING**

**Import Libraries**

In [1]:
import torch
import torch.nn.functional as F
torch.manual_seed(42)
print("PyTorch Version : " , torch.__version__)

PyTorch Version :  2.11.0+cpu


**Sample Input Embeddings**

In [3]:
tokens = ["I" , "love" , "python"]
embeddings = torch.tensor([
    [1.0 , 0.5 , 0.2 , 0.1] ,
    [0.8 , 1.0 , 0.4 , 0.3] ,
    [0.2 , 0.1 , 1.0 , 0.9]
])
print("Tokens : ")
print(tokens)
print("\nEmbeddings : ")
print(embeddings)

Tokens : 
['I', 'love', 'python']

Embeddings : 
tensor([[1.0000, 0.5000, 0.2000, 0.1000],
        [0.8000, 1.0000, 0.4000, 0.3000],
        [0.2000, 0.1000, 1.0000, 0.9000]])


**Create Query , Key and Value**

In [4]:
embedding_dim = embeddings.shape[1]
Wq = torch.randn(embedding_dim , embedding_dim)
Wk = torch.randn(embedding_dim , embedding_dim)
Wv = torch.randn(embedding_dim , embedding_dim)

Q = embeddings @ Wq
K = embeddings @ Wk
V = embeddings @ Wv

print("Query Matrix :\n")
print(Q)

print("\nKey Matrix :\n")
print(K)

print("\nValue Matrix : \n")
print(V)

Query Matrix :

tensor([[ 2.0429,  1.1438,  0.7238, -3.1123],
        [ 1.7007,  0.4469,  0.2899, -3.6218],
        [-0.9540,  1.3192, -0.9086, -1.2990]])

Key Matrix :

tensor([[ 1.4959e+00,  6.4302e-01, -6.1512e-05,  1.6328e+00],
        [ 9.9788e-01,  1.4817e+00,  5.7160e-01,  2.8241e+00],
        [ 1.3233e+00,  1.4099e+00,  3.6465e-01,  2.3646e+00]])

Value Matrix : 

tensor([[-1.6642, -0.6721, -0.3699,  1.1610],
        [-1.7941, -0.0864, -0.5953,  0.2125],
        [-2.9494,  2.6894, -2.0051, -0.7743]])


**Compute Attention Scores**

In [5]:
scores = Q @ K.T
print("Attention Scores : \n")
print(scores)

Attention Scores : 

tensor([[-1.2904, -4.6425, -2.7794],
        [-3.0823, -7.7034, -5.5776],
        [-2.6998, -3.1851, -2.8053]])


**Scale the Scores**

In [6]:
d_k = K.shape[1]
scaled_scores = scores / (d_k ** 0.5)
print("Scaled Scores : \n")
print(scaled_scores)

Scaled Scores : 

tensor([[-0.6452, -2.3213, -1.3897],
        [-1.5411, -3.8517, -2.7888],
        [-1.3499, -1.5925, -1.4026]])


**Apply Softmax**

In [7]:
attention_weights = F.softmax(scaled_scores , dim = -1)
print("Attention Weights : \n")
print(attention_weights)

Attention Weights : 

tensor([[0.6016, 0.1126, 0.2858],
        [0.7213, 0.0716, 0.2071],
        [0.3659, 0.2870, 0.3471]])


**Generate Context Vectors**

In [8]:
context = attention_weights @ V
print("Context Vectors : \n")
print(context)

Context Vectors : 

tensor([[-2.0461,  0.3545, -0.8626,  0.5012],
        [-1.9397,  0.0661, -0.7248,  0.6923],
        [-2.1475,  0.6627, -1.0021,  0.2171]])


**Apply Casual Mask**

In [9]:
sequence_length = len(tokens)
mask = torch.triu(
    torch.ones(sequence_length , sequence_length) ,
    diagonal = 1
)
mask = mask.bool()
print(mask)

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])


**Mask Future Tokens**

In [10]:
masked_scores = scaled_scores.masked_fill(mask , float('-inf'))
masked_attention = F.softmax(masked_scores , dim = -1)
print("Masked Attention Weights :\n")
print(masked_attention)

Masked Attention Weights :

tensor([[1.0000, 0.0000, 0.0000],
        [0.9097, 0.0903, 0.0000],
        [0.3659, 0.2870, 0.3471]])


**Context After Masking**

In [11]:
masked_context = masked_attention @ V
print("Masked Context : \n")
print(masked_context)

Masked Context : 

tensor([[-1.6642, -0.6721, -0.3699,  1.1610],
        [-1.6759, -0.6192, -0.3903,  1.0754],
        [-2.1475,  0.6627, -1.0021,  0.2171]])


**Compare Both**

In [13]:
print("Attention Without Mask : \n")
print(attention_weights)

print("\nAttention With Casual Mask :\n")
print(masked_attention)

Attention Without Mask : 

tensor([[0.6016, 0.1126, 0.2858],
        [0.7213, 0.0716, 0.2071],
        [0.3659, 0.2870, 0.3471]])

Attention With Casual Mask :

tensor([[1.0000, 0.0000, 0.0000],
        [0.9097, 0.0903, 0.0000],
        [0.3659, 0.2870, 0.3471]])


**Display Attention for Each Token**

In [14]:
for i, token in enumerate(tokens):
    print(f"\n{token} attends to :")

    for j, other in enumerate(tokens):
        print(f"{other:<10}: {masked_attention[i][j]:.4f}")


I attends to :
I         : 1.0000
love      : 0.0000
python    : 0.0000

love attends to :
I         : 0.9097
love      : 0.0903
python    : 0.0000

python attends to :
I         : 0.3659
love      : 0.2870
python    : 0.3471
